# Heatmap - Interactive Development

Create publication-ready heatmaps with customizable parameters.

**Workflow:**
1. Run Setup (Section 1)
2. Load Data (Section 2)
3. Adjust Parameters (Section 3)
4. Generate Heatmap (Section 4) - **Re-run this cell to see changes**
5. Export when satisfied (Section 5)

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our heatmap module
from heatmap_plot import HeatmapPlotter, load_integrated_results

# High-resolution plots
%config InlineBackend.figure_format = 'retina'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

print("✓ Setup complete")

## 2. Load Data

In [ ]:
# Specify data path
DATA_PATH = '../output/4_groups/integrated_results_cortex.csv'

# Load full integrated results
df_full = pd.read_csv(DATA_PATH)
print(f"✓ Loaded {len(df_full)} m/z bins from {Path(DATA_PATH).name}")

# Extract sample intensity data
# Auto-detect sample columns (pattern: group_number)
sample_cols = [col for col in df_full.columns 
               if '_' in col and col.split('_')[-1].isdigit()
               and not col.startswith('log2FC')
               and not col.startswith('FC_')
               and not col.endswith('_mean')
               and not col.endswith('_sd')]

print(f"\nDetected {len(sample_cols)} samples: {sample_cols}")

# Create data matrix for heatmap
data = df_full.set_index('m_z_bin')[sample_cols]

print(f"\nData shape: {data.shape[0]} m/z bins × {data.shape[1]} samples")
print(f"\nFirst few rows:")
display(data.head())

## 3. Configure Heatmap Parameters

**Adjust these parameters and re-run Section 4 to see changes**

In [ ]:
# ============================================================
# DATA PROCESSING
# ============================================================

# Normalization method
NORMALIZATION = 'zscore'  # 'zscore', 'minmax', 'robust', 'none'
                          # zscore: mean=0, std=1 (recommended for heatmaps)
                          # minmax: scale to 0-1 range
                          # robust: use median and IQR (robust to outliers)
                          # none: use raw values

# Log transformation (apply before normalization)
LOG_TRANSFORM = False     # True: apply log2(x+1) transformation

# Normalization axis
NORM_AXIS = 0             # 0: normalize each row (across samples)
                          # 1: normalize each column (across m/z bins)

# ============================================================
# CLUSTERING SETTINGS
# ============================================================

# Enable/disable clustering
CLUSTER_ROWS = True       # Cluster m/z bins (rows)
CLUSTER_COLS = True       # Cluster samples (columns)

# Clustering algorithm
CLUSTER_METHOD = 'average'  # Linkage method:
                            # 'average': UPGMA (recommended)
                            # 'complete': maximum distance
                            # 'single': minimum distance
                            # 'ward': minimize variance (only with euclidean)

# Distance metric
CLUSTER_METRIC = 'euclidean'  # Distance metric:
                              # 'euclidean': Euclidean distance (recommended)
                              # 'correlation': 1 - Pearson correlation
                              # 'cosine': Cosine distance
                              # 'cityblock': Manhattan distance

# ============================================================
# DENDROGRAM SETTINGS
# ============================================================

# Show/hide dendrograms
SHOW_ROW_DENDROGRAM = True   # Show row dendrogram
SHOW_COL_DENDROGRAM = True   # Show column dendrogram

# Dendrogram size (relative to heatmap)
DENDROGRAM_RATIO = 0.15      # Size ratio (0.1-0.3 recommended)

# ============================================================
# HEATMAP APPEARANCE
# ============================================================

# Figure size
FIG_WIDTH = 12            # Width in inches
FIG_HEIGHT = 10           # Height in inches

# Color scheme
COLORMAP = 'RdBu_r'       # Colormap name:
                          # 'RdBu_r': Red-Blue (diverging, recommended)
                          # 'viridis': Yellow-Green-Blue (sequential)
                          # 'coolwarm': Blue-Red (diverging)
                          # 'YlOrRd': Yellow-Orange-Red (sequential)
                          # 'PiYG': Pink-Yellow-Green (diverging)

# Color scale
CENTER_VALUE = 0          # Center colormap at this value (for diverging colormaps)
                          # Set to None for sequential colormaps
VMIN = None               # Minimum value (None = auto)
VMAX = None               # Maximum value (None = auto)

# Cell appearance
SHOW_VALUES = False       # Annotate cells with values
VALUE_FORMAT = '.2f'      # Format for cell values
LINE_WIDTH = 0            # Width of lines between cells (0 = no lines)
LINE_COLOR = 'white'      # Color of lines between cells

# Labels
SHOW_XTICKLABELS = True   # Show sample names
SHOW_YTICKLABELS = True   # Show m/z bin values
XLABEL = 'Samples'
YLABEL = 'm/z bins'
TITLE = None              # Plot title (None = auto-generate)
CBAR_LABEL = 'Z-score'    # Colorbar label

# ============================================================
# FILTERING (Optional)
# ============================================================

# Filter by significance (requires p_adj column)
FILTER_SIGNIFICANT = False  # Only show significant m/z bins
FDR_THRESHOLD = 0.05        # FDR threshold for filtering

# Filter by fold change (requires log2FC columns)
FILTER_BY_FC = False        # Only show m/z bins with large fold changes
FC_THRESHOLD = 1.0          # Minimum |log2FC| to include

# Top N features
TOP_N_FEATURES = None       # Show only top N most variable features (None = all)

print("✓ Parameters configured")
print(f"  Normalization: {NORMALIZATION}")
print(f"  Log transform: {LOG_TRANSFORM}")
print(f"  Row clustering: {CLUSTER_ROWS} ({CLUSTER_METHOD}, {CLUSTER_METRIC})")
print(f"  Column clustering: {CLUSTER_COLS}")
print(f"  Colormap: {COLORMAP}")

## 4. Generate Heatmap

**Re-run this cell after changing parameters in Section 3**

In [ ]:
# Apply filters if requested
data_filtered = data.copy()

if FILTER_SIGNIFICANT and 'p_adj' in df_full.columns:
    sig_bins = df_full[df_full['p_adj'] < FDR_THRESHOLD]['m_z_bin'].values
    data_filtered = data_filtered.loc[data_filtered.index.isin(sig_bins)]
    print(f"Filtered to {len(data_filtered)} significant m/z bins (FDR < {FDR_THRESHOLD})")

if FILTER_BY_FC:
    # Find m/z bins with any comparison exceeding FC threshold
    fc_cols = [col for col in df_full.columns if col.startswith('log2FC_')]
    if fc_cols:
        max_abs_fc = df_full[fc_cols].abs().max(axis=1)
        high_fc_bins = df_full[max_abs_fc >= FC_THRESHOLD]['m_z_bin'].values
        data_filtered = data_filtered.loc[data_filtered.index.isin(high_fc_bins)]
        print(f"Filtered to {len(data_filtered)} m/z bins with |log2FC| >= {FC_THRESHOLD}")

if TOP_N_FEATURES is not None and TOP_N_FEATURES < len(data_filtered):
    # Select top N most variable features
    variance = data_filtered.var(axis=1)
    top_features = variance.nlargest(TOP_N_FEATURES).index
    data_filtered = data_filtered.loc[top_features]
    print(f"Selected top {TOP_N_FEATURES} most variable m/z bins")

print(f"\nFinal data shape: {data_filtered.shape[0]} m/z bins × {data_filtered.shape[1]} samples")

# Create HeatmapPlotter instance
plotter = HeatmapPlotter(data_filtered)

# Normalize data
plotter.normalize_data(
    method=NORMALIZATION,
    axis=NORM_AXIS,
    log_transform=LOG_TRANSFORM
)

# Generate heatmap
fig = plotter.plot_heatmap(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    cmap=COLORMAP,
    center=CENTER_VALUE,
    show_row_dendrogram=SHOW_ROW_DENDROGRAM,
    show_col_dendrogram=SHOW_COL_DENDROGRAM,
    dendrogram_ratio=DENDROGRAM_RATIO,
    row_cluster=CLUSTER_ROWS,
    col_cluster=CLUSTER_COLS,
    cluster_method=CLUSTER_METHOD,
    cluster_metric=CLUSTER_METRIC,
    vmin=VMIN,
    vmax=VMAX,
    xlabel=XLABEL,
    ylabel=YLABEL,
    title=TITLE if TITLE else f'Heatmap: {Path(DATA_PATH).stem}',
    annot=SHOW_VALUES,
    fmt=VALUE_FORMAT,
    linewidths=LINE_WIDTH,
    linecolor=LINE_COLOR,
    xticklabels=SHOW_XTICKLABELS,
    yticklabels=SHOW_YTICKLABELS,
    cbar_label=CBAR_LABEL
)

plt.show()

# Print summary
print(f"\n{'='*60}")
print(f"HEATMAP SUMMARY")
print(f"{'='*60}")
print(f"Data file: {Path(DATA_PATH).name}")
print(f"Features (m/z bins): {data_filtered.shape[0]}")
print(f"Samples: {data_filtered.shape[1]}")
print(f"Normalization: {NORMALIZATION}")
print(f"Log transform: {LOG_TRANSFORM}")
print(f"Row clustering: {CLUSTER_ROWS}")
print(f"Column clustering: {CLUSTER_COLS}")
print(f"{'='*60}")

## 5. Export Figure and Data

Run this cell when you're satisfied with the heatmap to save it.

In [ ]:
# Output settings
OUTPUT_DIR = Path('../output/4_groups/heatmaps')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# File naming
roi_name = Path(DATA_PATH).stem.replace('integrated_results_', '')
base_name = f'heatmap_{roi_name}'

# Save figure as PNG (high resolution)
png_path = OUTPUT_DIR / f'{base_name}.png'
fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✓ Saved PNG: {png_path}")

# Save figure as PDF (vector, for publications)
pdf_path = OUTPUT_DIR / f'{base_name}.pdf'
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
print(f"✓ Saved PDF: {pdf_path}")

# Optional: Save as SVG (vector, editable in Illustrator)
# svg_path = OUTPUT_DIR / f'{base_name}.svg'
# fig.savefig(svg_path, bbox_inches='tight', facecolor='white')
# print(f"✓ Saved SVG: {svg_path}")

# Export data
data_export_path = OUTPUT_DIR / f'{base_name}_data'
plotter.export_data(
    str(data_export_path),
    include_original=True,
    include_normalized=True,
    include_clustered=True
)

print(f"\n✓ All files saved to: {OUTPUT_DIR}")

## 6. Quick Reference: Common Parameter Values

### Normalization Methods
```python
NORMALIZATION = 'zscore'   # Z-score (mean=0, std=1) - recommended
NORMALIZATION = 'minmax'   # Min-max scaling (0 to 1)
NORMALIZATION = 'robust'   # Robust scaling (median, IQR)
NORMALIZATION = 'none'     # No normalization
```

### Clustering Methods
```python
# Linkage methods
CLUSTER_METHOD = 'average'   # UPGMA (recommended)
CLUSTER_METHOD = 'complete'  # Maximum distance
CLUSTER_METHOD = 'single'    # Minimum distance
CLUSTER_METHOD = 'ward'      # Minimize variance (euclidean only)

# Distance metrics
CLUSTER_METRIC = 'euclidean'    # Euclidean distance (recommended)
CLUSTER_METRIC = 'correlation'  # 1 - Pearson correlation
CLUSTER_METRIC = 'cosine'       # Cosine distance
CLUSTER_METRIC = 'cityblock'    # Manhattan distance
```

### Color Schemes
```python
# Diverging colormaps (for normalized data)
COLORMAP = 'RdBu_r'      # Red-Blue (recommended)
COLORMAP = 'coolwarm'    # Blue-Red
COLORMAP = 'PiYG'        # Pink-Yellow-Green
COLORMAP = 'RdYlBu_r'    # Red-Yellow-Blue

# Sequential colormaps (for non-negative data)
COLORMAP = 'viridis'     # Yellow-Green-Blue
COLORMAP = 'YlOrRd'      # Yellow-Orange-Red
COLORMAP = 'Blues'       # White-Blue
COLORMAP = 'Reds'        # White-Red
```

### Common Filtering Scenarios
```python
# Show only significant features
FILTER_SIGNIFICANT = True
FDR_THRESHOLD = 0.05

# Show only features with large fold changes
FILTER_BY_FC = True
FC_THRESHOLD = 1.0  # |log2FC| >= 1 (2-fold change)

# Show top 50 most variable features
TOP_N_FEATURES = 50
```

## 7. Advanced: Custom Sample Grouping

Group samples by experimental condition for better visualization.

In [ ]:
# Example: Reorder samples by group
# Extract group names from sample columns
sample_groups = {col: col.rsplit('_', 1)[0] for col in sample_cols}

# Sort samples by group
sorted_samples = sorted(sample_cols, key=lambda x: (sample_groups[x], x))

# Reorder data
data_grouped = data[sorted_samples]

print("Sample order by group:")
for sample in sorted_samples:
    print(f"  {sample} ({sample_groups[sample]})")

# Create heatmap with grouped samples
plotter_grouped = HeatmapPlotter(data_grouped)
plotter_grouped.normalize_data(method=NORMALIZATION, axis=NORM_AXIS, log_transform=LOG_TRANSFORM)

fig_grouped = plotter_grouped.plot_heatmap(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    cmap=COLORMAP,
    center=CENTER_VALUE,
    show_row_dendrogram=SHOW_ROW_DENDROGRAM,
    show_col_dendrogram=False,  # Disable column clustering to preserve grouping
    row_cluster=CLUSTER_ROWS,
    col_cluster=False,
    cluster_method=CLUSTER_METHOD,
    cluster_metric=CLUSTER_METRIC,
    title='Heatmap with Grouped Samples',
    cbar_label=CBAR_LABEL
)

plt.show()